In [127]:
import pandas as pd
import random
import ast

In [151]:
model = "mBERT"
data = "baseline"
sl = "lenient"

In [152]:
def get_sample(df, output_file):
    df = df.copy()
    df["categories"] = df["categories"].apply(ast.literal_eval)
    df["levels"] = df["levels"].apply(ast.literal_eval)

    df["predicted_categories"] = df["predicted_categories"].apply(ast.literal_eval)
    df["predicted_level"] = df["predicted_level"].apply(ast.literal_eval)
    
    disjoint = df[df.apply(lambda row: is_disjoint(row["categories"], row["levels"]), axis=1)]
    
    print(len(disjoint), "disjoint samples found.")

    # Choose 40 random samples from the disjoint set
    sample = disjoint.sample(
        n=min(40, len(disjoint)),
        random_state=42
    )

    rows = []

    for _, row in disjoint.iterrows():
        idx = row.name
        conv = row["conversation_id"]

        context = df.iloc[max(0, idx-3):idx+1].copy()
        context = context[context["conversation_id"] == conv].copy()
        
        context["sample_turn"] = False
        context.loc[row.name, "sample_turn"] = True

        rows.append(context)

    result = pd.concat(rows)

    result.to_csv(output_file, index=False)

    return result

In [156]:
def is_disjoint(categories, levels):
    
    for cat, level in zip(categories, levels):
        if cat not in [None, "None"] and level in [None, "None"]:
            return True
    return False

In [ ]:
input_file = ""
prediction_df = pd.read_csv(input_file)

output_file = ""

In [158]:
disjoint_results = get_sample(prediction_df, output_file)

861 disjoint samples found.


In [159]:
def normalise(x):
    if isinstance(x, tuple):
        return list(x)
    return x

In [160]:
def analysis(df):
    sample = df[df["sample_turn"]].copy()

    sample["predicted_categories"] = sample["predicted_categories"].apply(normalise)
    sample["predicted_level"] = sample["predicted_level"].apply(normalise)

    correct_categories = 0
    extra_categories = 0

    incorrect_levels = 0
    avg_predicted_levels = []
    avg_gold_levels = []

    for _, row in sample.iterrows():
        gold_cats = row["categories"]
        gold_levels = row["levels"]
        
        pred_cats = row["predicted_categories"]
        pred_levels = row["predicted_level"]

        for gold_cat in gold_cats:
            if gold_cat in [None, "None"]:
                continue
            if gold_cat in pred_cats:
                correct_categories += 1

        for pred_cat in pred_cats:
            if pred_cat not in [None, "None"] and pred_cat not in gold_cats:
                extra_categories += 1
                
        for gold_cat, gold_level in zip(gold_cats, gold_levels):            
            if gold_cat in pred_cats:
                idx = pred_cats.index(gold_cat)
                if pred_levels[idx] not in [None, "None"] and gold_level in [None, "None"]:
                    incorrect_levels += 1
                    avg_predicted_levels.append(int(pred_levels[idx]))
                if pred_levels[idx] in [None, "None"] and gold_level not in [None, "None"]:
                    incorrect_levels += 1
                    avg_gold_levels.append(gold_level)
                    
                    
    print(f"Correct categories: {correct_categories}")
    print(f"Extra predicted categories: {extra_categories}")
    print(f"Levels incorrectly predicted: {incorrect_levels}")
    if avg_predicted_levels:
        print(f"Average predicted level: {sum(avg_predicted_levels)/len(avg_predicted_levels):.2f}")
    if avg_gold_levels:
        print(f"Average gold level: {sum(avg_gold_levels)/len(avg_gold_levels):.2f}")
            

In [161]:
print("Analysis results")
analysis(disjoint_results)

Analysis results
Correct categories: 514
Extra predicted categories: 193
Levels incorrectly predicted: 514
Average predicted level: 2.00
